# PHYB57 — Number Representation Systems & Floating-Point Arithmetic

Converted from **PHYB57_C9.pdf**. The lecture content is organized by the original slide order. Code examples are editable Python cells; some cells use NumPy.


## Slide 1 — Understanding Number Representation Systems

### Decimal System (Base-10)

- The most familiar number system, composed of digits 0–9.
- Numbers are represented as a sum of powers of 10.
- Example: `147.3` in base-10 expands to `1×10² + 4×10¹ + 7×10⁰ + 3×10⁻¹`.

### Base-N Systems

- Any number system can use a base *N*, where *N* is the number of unique digits in the system.
- Example (Base-3): `121 (base-3) = 1×3² + 2×3¹ + 1×3⁰ = 16 (base-10)`.


In [1]:
# Decimal (base-10) expansion of 147.3
value = 1*10**2 + 4*10**1 + 7*10**0 + 3*10**-1
print("147.3 base-10 =", value)

# Base-3 expansion of 121 -> decimal
base3 = 1*3**2 + 2*3**1 + 1*3**0
print("121 base-3 =", base3, "base-10")

# General base-N conversion helper (integer part)
def base_to_decimal(digits, base):
    return sum(d * base**i for i, d in enumerate(reversed(digits)))

print("121 base-3 =", base_to_decimal([1, 2, 1], 3))


147.3 base-10 = 147.3
121 base-3 = 16 base-10
121 base-3 = 16


## Slide 2 — Binary Numbers (Base-2)

- A key number system for computers, using only digits 0 and 1.
- Each digit is a coefficient of a power of 2, also known as a **bit**.

### Conversion Examples

- `11 (base-10) = 1011 (base-2)`
- `37 (base-10) = 100101 (base-2)`


In [2]:
# Decimal -> binary
print(f"11  (base-10) = {bin(11)[2:]} (base-2)")
print(f"37  (base-10) = {bin(37)[2:]} (base-2)")

# Binary -> decimal
print(f"1011   (base-2) = {int('1011', 2)}  (base-10)")
print(f"100101 (base-2) = {int('100101', 2)}  (base-10)")


11  (base-10) = 1011 (base-2)
37  (base-10) = 100101 (base-2)
1011   (base-2) = 11  (base-10)
100101 (base-2) = 37  (base-10)


## Slide 3 — Binary Arithmetic

### Binary Addition and Multiplication

- **Addition example:** `37 + 17` in binary is `100101 + 10001 = 110110`, which is 54 in decimal.
- **Multiplication example:** `37 × 17` in binary is `100101 × 10001 = 1001110101`, which is 629 in decimal.

### Why Binary is Useful in Computers

- Binary operations are implemented using logic gates (AND, OR, NOT), which computers execute quickly.

### Limitations

- Binary numbers are stored in fixed bit lengths (e.g., 32-bit systems), restricting the range of representable numbers.


In [ ]:
a, b = 37, 17
print(f"Addition:       {a} + {b} = {a + b:>3}  ->  {bin(a)} + {bin(b)} = {bin(a + b)}")
print(f"Multiplication: {a} x {b} = {a * b:>3}  ->  {bin(a)} x {bin(b)} = {bin(a * b)}")

# Verify the slide results
assert int('100101', 2) + int('10001', 2) == 54
assert int('100101', 2) * int('10001', 2) == 629
print("Slide results verified: 110110 = 54, 1001110101 = 629")


## Slide 4 — Floating Point Numbers

### What are Floating Point Numbers?

- Used to represent real numbers in a computer, providing a wide range of values with a fixed number of bits.
- Crucial for scientific and engineering calculations that need high precision and a broad range of values.

### Components of Floating Point Numbers

- **Sign (s):** whether the number is positive or negative.
- **Exponent (e):** the power of 2, indicating the scale of the number.
- **Fraction (f):** the significand or mantissa, a value between 1 and 2, indicating the precision.

### IEEE 754 Double Precision Format

- Python maps floats to IEEE 754 double precision, which uses **64 bits**:
  - 1 bit for the sign, 11 bits for the exponent, 52 bits for the fraction.
- The exponent has a **bias of 1023**, so `e - 1023` normalizes the exponent.


In [ ]:
import struct

def float_bits(x):
    """Return the sign, biased exponent, and fraction bits of a double."""
    bits = struct.unpack('>Q', struct.pack('>d', x))[0]
    s = (bits >> 63) & 0x1
    e = (bits >> 52) & 0x7FF
    f = bits & 0xFFFFFFFFFFFFF
    return s, e, f

for x in [1.0, -1.5, 0.5, 1024.0]:
    s, e, f = float_bits(x)
    print(f"{x:>8}: sign={s}, exponent={e} (biased), fraction bits={f:#018x}")


## Slide 5 — Calculations and Precision

### Representation of a Floating Point Number

- A float can be represented as `n = (-1)^s × 2^(e-1023) × (1 + f)`.

### Example

- For the binary number `1 10000000010 1000…0`:
  - Sign `s = 1` (negative)
  - Exponent `e = 1026` (decimal exponent: 3)
  - Fraction `f = 1.5`

### Precision Considerations

- The gap between two adjacent floating-point numbers increases with magnitude.
- `numpy.spacing(x)` returns the distance between adjacent floats, which grows as `x` increases.


In [ ]:
import numpy as np

# Reconstruct the slide's example: n = (-1)^s * 2^(e-1023) * (1 + f)
s, e, f = 1, 1026, 1.5
n = (-1)**s * 2**(e - 1023) * f
print("Reconstructed float:", n)  # -> -12.0

# The gap between adjacent floats grows with magnitude
for x in [1.0, 1e10, 1e100, 1e300]:
    print(f"spacing({x}) = {np.spacing(x)}")


## Slide 6 — Special Cases and Limitations

### Special Floating Point Cases

- **Subnormal numbers:** occur when the exponent is zero, allowing representation of very small numbers.
- **Infinity and NaN (Not a Number):** occur when the exponent is all 1s (2047). If the fraction is zero the result is infinity; if not, it is NaN.

### Handling Overflow and Underflow

- **Overflow:** a result exceeds the largest representable number; Python handles it as `inf`.
- **Underflow:** a result is smaller than the smallest representable subnormal; Python handles it as `0`.


In [1]:
import numpy as np

# Overflow to infinity
print("Overflow to infinity:", 1e308 * 10)   # -> inf

# NaN arises from an indeterminate operation
print("inf * 0 =", np.inf * 0)               # -> nan

# Underflow to zero
print("Underflow to zero:", 1e-308 / 1e308)  # -> 0.0

# Smallest positive subnormal number
print("Smallest subnormal:", np.nextafter(0.0, 1.0))


Overflow to infinity: inf
inf * 0 = nan
Underflow to zero: 0.0
Smallest subnormal: 5e-324


## Slide 7 — Understanding Round-Off Errors

### What are Round-Off Errors?

- Round-off error occurs because numbers are approximated when stored in a computer's memory.
- Computers represent floating-point numbers in binary, which cannot always perfectly represent decimal numbers.
- The difference between the actual value and its stored approximation is called a **round-off error**.

### Representation Error

- A common cause of round-off errors.
- Example: `π` (3.14159…) or `1/3` (0.33333…) cannot be represented perfectly, leading to small but cumulative errors.

### Rounding Multiple Times

- Successive rounding can exacerbate errors. Rounding `4.845` to two decimals gives `4.85`, then to one decimal gives `4.9` — a larger cumulative error than rounding directly to one decimal (`4.8`).


In [ ]:
from decimal import Decimal, ROUND_HALF_UP

# Representation error: some decimal values are not exact in binary floats
print("1/3 stored as:", repr(1 / 3))
print("pi stored as:", repr(3.141592653589793))

# Rounding multiple times can increase error
x = Decimal("4.845")
two_dec = x.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)   # 4.85
one_dec = two_dec.quantize(Decimal("0.1"), rounding=ROUND_HALF_UP)  # 4.9
direct  = x.quantize(Decimal("0.1"), rounding=ROUND_HALF_UP)      # 4.8
print("Round to 2 then 1 decimal:", two_dec, "->", one_dec)
print("Round directly to 1 decimal:", direct)


## Slide 8 — Examples of Round-Off Errors in Python

### Floating Point Arithmetic Issues

- Operations in Python demonstrate how floating-point numbers accumulate round-off errors.
- Example: `4.9 - 4.845` in Python yields `0.055000000000000604`, not exactly `0.055`.
- Example: `0.1 + 0.2 + 0.3 != 0.6` due to rounding approximations.

### Accumulation of Round-Off Error

- Errors accumulate over repetitive calculations.
- Adding and subtracting `1/3` multiple times:
  - Done once: `1 + 1/3 - 1/3` evaluates to `1`.
  - Repeated 100 times: the result becomes `1.0000000000000002`.
- As the number of repetitions increases, the error increases proportionally.


In [ ]:
# Floating-point arithmetic is not exact decimal arithmetic
print("4.9 - 4.845 =", 4.9 - 4.845)          # 0.055000000000000604
print("0.1 + 0.2 + 0.3 =", 0.1 + 0.2 + 0.3)  # 0.6000000000000001
print("0.6 =", 0.6)
print("0.1 + 0.2 + 0.3 == 0.6 ->", 0.1 + 0.2 + 0.3 == 0.6)


In [ ]:
# Error accumulates with repetition
x = 1.0
for _ in range(100):
    x = x + 1/3 - 1/3
print("1 + 100*(1/3 - 1/3) =", x)  # 1.0000000000000002
